## Split golden data into Train Validation and Test csv's

In [1]:
import pandas as pd
import numpy as np

In [5]:
# Grab file
file_path = "../../ais_data/golden_datasetv4.csv"
init_df = pd.read_csv(file_path)

In [6]:
print(init_df.dtypes)

Unnamed: 0               int64
MSG_TYPE                 int64
MMSI                     int64
NAME                    object
IMO_NUMBER             float64
CALL_SIGN               object
LAT_AVG                float64
LON_AVG                float64
PERIOD                  object
SPEED_KNOTS            float64
COG_DEG                float64
HEADING_DEG            float64
NAV_STATUS             float64
NAV_SENSOR             float64
SHIP_AND_CARGO_TYPE      int64
DRAUGHT                float64
DRAUGHT.1              float64
DIM_BOW                  int64
DIM_STERN                int64
DIM_PORT                 int64
DIM_STARBOARD            int64
DESTINATION             object
MMSI_COUNTRY_CD         object
RECEIVER                object
BEAM                     int64
LENGTH                   int64
CHANNEL_SIDE            object
time_diff              float64
cog_diff               float64
new_voyage                bool
voyage_id               object
geometry                object
in_chann

In [7]:
# Only take columns needed for TCN Neural Network
df = init_df[['MMSI', 'LAT_AVG', 'LON_AVG', 'PERIOD', 'SPEED_KNOTS', 'COG_DEG', 'HEADING_DEG', 'voyage_id']]

# Make a datetime type column
df['TIME'] = pd.to_datetime(df['PERIOD'], errors='coerce')
df = df.rename(columns = {'LAT_AVG':'LAT', 'LON_AVG':'LON', 'SPEED_KNOTS':'SPEED', 'COG_DEG':'COG', 'HEADING_DEG':'HEADING'})

print("datetime check", df['TIME'].dtypes)

# drop all NA's
print("pre-NA drop", len(df))
df = df.dropna()
print("post-NA drop", len(df))

datetime check datetime64[ns]
pre-NA drop 21148
post-NA drop 20584


C:\Users\LAKwon\AppData\Local\Temp\ipykernel_3968\768074335.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['TIME'] = pd.to_datetime(df['PERIOD'], errors='coerce')


In [8]:
# make delta time, lat, and lon columns for TCN
df['dt'] = df['TIME'].diff().dt.total_seconds()
df['dlat'] = df['LAT'].diff()
df['dlon'] = df['LON'].diff()

# sort values by vessel and time
df = df.sort_values(["MMSI", "TIME"]).reset_index(drop=True)

# fast way to change each new voyage delta time, lat, and lon from NaN to 0
mask = df['voyage_id'] != df['voyage_id'].shift()
df.loc[mask, ['dt', 'dlat', 'dlon']] = 0

# older slower way
'''for voyage in range(0, len(df)):
    if voyage > 0 and df.loc[voyage-1, "dt"] != df.loc[voyage, "dt"]:
        df.loc[voyage, ["dt", "dlat", "dlon"]] = 0'''

# change cog and heading to cos and sin for TCN
df['cog_rad'] = np.deg2rad(df['COG'])
df['cog_sin'] = np.sin(df['cog_rad'])
df['cog_cos'] = np.cos(df['cog_rad'])

df['hdg_rad'] = np.deg2rad(df['HEADING'])
df['hdg_sin'] = np.sin(df['hdg_rad'])
df['hdg_cos'] = np.cos(df['hdg_rad'])



df.head()

,MMSI,LAT,LON,PERIOD,SPEED,COG,HEADING,voyage_id,TIME,dt,dlat,dlon,cog_rad,cog_sin,cog_cos,hdg_rad,hdg_sin,hdg_cos
0,205089000,22.547895,-78.097770,2023-09-27 14:10:00,12.9,121.1,121.0,20_205089000,2023-09-27 14:10:00,0.0,0.000000,0.000000,2.113594,0.856267,-0.516533,2.111848,0.857167,-0.515038
1,205089000,22.523830,-78.055422,2023-09-27 14:20:00,12.9,121.6,122.0,20_205089000,2023-09-27 14:20:00,600.0,-0.024065,0.042348,2.122320,0.851727,-0.523986,2.129302,0.848048,-0.529919
2,205089000,22.517936,-78.045039,2023-09-27 14:25:00,12.8,121.4,122.0,20_205089000,2023-09-27 14:25:00,300.0,-0.005894,0.010383,2.118830,0.853551,-0.521010,2.129302,0.848048,-0.529919
3,205089000,22.507930,-78.027377,2023-09-27 14:30:00,12.8,122.0,122.0,20_205089000,2023-09-27 14:30:00,300.0,-0.010006,0.017662,2.129302,0.848048,-0.529919,2.129302,0.848048,-0.529919
4,205089000,22.479852,-77.977003,2023-09-27 14:45:00,12.6,121.3,123.0,20_205089000,2023-09-27 14:45:00,900.0,-0.028078,0.050374,2.117084,0.854459,-0.519519,2.146755,0.838671,-0.544639


In [9]:
# Reduce columns again
training_df = df[["MMSI", "voyage_id", "dt", "cog_sin", "cog_cos", "hdg_sin", "hdg_cos", "dlat", "dlon"]]

training_df.head()

,MMSI,voyage_id,dt,cog_sin,cog_cos,hdg_sin,hdg_cos,dlat,dlon
0,205089000,20_205089000,0.0,0.856267,-0.516533,0.857167,-0.515038,0.000000,0.000000
1,205089000,20_205089000,600.0,0.851727,-0.523986,0.848048,-0.529919,-0.024065,0.042348
2,205089000,20_205089000,300.0,0.853551,-0.521010,0.848048,-0.529919,-0.005894,0.010383
3,205089000,20_205089000,300.0,0.848048,-0.529919,0.848048,-0.529919,-0.010006,0.017662
4,205089000,20_205089000,900.0,0.854459,-0.519519,0.838671,-0.544639,-0.028078,0.050374


In [10]:
# List of vessels
vessels = df['MMSI'].unique()

# Shuffle for randomness (VERY important)
rng = np.random.default_rng(42)
rng.shuffle(vessels)

n_total = len(vessels)

# ~90% train and ~10% testing
test_split = int(n_total * 0.9)
train_val_vessels = vessels[:test_split]
test_vessels      = vessels[test_split:]

# Split train into ~75% train and ~15% val
val_frac_of_train = 0.2   # ~20% of the ~90%
val_split = int(len(train_val_vessels) * (1 - val_frac_of_train))

train_vessels = train_val_vessels[:val_split]
val_vessels   = train_val_vessels[val_split:]

# Create DataFrames
df_train = df[df['MMSI'].isin(train_vessels)].copy()
df_val   = df[df['MMSI'].isin(val_vessels)].copy()
df_test  = df[df['MMSI'].isin(test_vessels)].copy()

# Diagnostics
print("Train %:", len(df_train)/len(df))
print("Val %:",   len(df_val)/len(df))
print("Test %:",  len(df_test)/len(df))

print("Train rows:", len(df_train))
print("Val rows:",   len(df_val))
print("Test rows:",  len(df_test))

Train %: 0.7366401088223863
Val %: 0.15648076175670422
Test %: 0.10687912942090945
Train rows: 15163
Val rows: 3221
Test rows: 2200


In [11]:
# More Diagnostics
print("number of Train vessels:", df_train["MMSI"].nunique())
print("number of Val vessels:", df_val["MMSI"].nunique())
print("number of Test vessels:", df_test["MMSI"].nunique())

print("number of Train voyages:", df_train["voyage_id"].nunique())
print("number of Val voyages:", df_val["voyage_id"].nunique())
print("number of Test voyages:", df_test["voyage_id"].nunique())

number of Train vessels: 621
number of Val vessels: 156
number of Test vessels: 87
number of Train voyages: 1371
number of Val voyages: 277
number of Test voyages: 200


In [12]:
# Create csv's
df_train.to_csv("gold_train_data.csv", index=False)
df_val.to_csv("gold_val_data.csv", index=False)
df_test.to_csv("gold_test_data.csv", index=False)